<a href="https://colab.research.google.com/github/Akansha102005/DTSC5565/blob/main/Prompt_vs_Agent_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt vs Agent

## Getting Started

* Colab - get notebook from gitmystuff
* Save a Copy in Drive
* Remove Copy of
* Edit NAME and TASK IDEA
* Clean up Colab Notebooks folder
* Submit shared link

### Your details

In [ ]:
# task idea must be taken from one of the prompts we worked on from Sommerville
  NAME = "Akansha"
MY_TASK_IDEA = "Create a test plan for a campus peer tutoring booking system"


## Setup
Add your Groq key as a **Secret**: click the key icon on the left, **Add new secret**, name it `GROQ_API_KEY`, paste your key as the value, and switch **Notebook access** ON. Then run the cell below.

In [ ]:
# SETUP: run this cell once. It installs the Groq library, finds your key, and defines ask_groq().
import os, re, time, json, copy

try:
    import groq
except ImportError:
    get_ipython().system("pip install -q groq")
    import groq

GROQ_MODEL = "openai/gpt-oss-120b"   # if Groq stops offering this model, ask_groq() picks a working one automatically

def _find_key():
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        return os.environ.get("GROQ_API_KEY")

GROQ_KEY = _find_key()
if not GROQ_KEY:
    print("NO KEY FOUND. Click the key icon (Secrets) on the left, add a secret named GROQ_API_KEY,")
    print("paste your Groq key as its value, and switch ON 'Notebook access'. Then run this cell again.")
    print("(Never paste your key into a code cell: shared notebooks show everything in them.)")
else:
    print("OK: key found (not showing it).")

_client = None

def ask_groq(messages, max_tokens=2000):
    """Send a list of {"role": ..., "content": ...} messages to Groq. Returns the reply text."""
    global _client, GROQ_MODEL
    if not GROQ_KEY:
        raise RuntimeError("No Groq key yet. See the SETUP cell above.")
    if _client is None:
        _client = groq.Groq(api_key=GROQ_KEY, max_retries=1, timeout=120)
    for attempt in (1, 2, 3):
        try:
            r = _client.chat.completions.create(model=GROQ_MODEL, messages=messages,
                                                max_tokens=max_tokens, temperature=0.2)
            return (r.choices[0].message.content or "").strip() or "(empty reply)"
        except groq.RateLimitError:
            if attempt == 3:
                raise RuntimeError("Groq's free tier says you are going too fast. Wait a minute and try again.")
            print(f"[wait] Groq asked us to slow down; waiting 20 seconds (try {attempt} of 3)...")
            time.sleep(20)
        except groq.AuthenticationError:
            raise RuntimeError("Groq rejected your key. Check the GROQ_API_KEY secret.")
        except groq.NotFoundError:
            # The model name was retired. Ask Groq what is available and pick a chat model.
            ids = [m.id for m in _client.models.list().data]
            chat = [i for i in ids if not any(w in i.lower() for w in ("whisper", "guard", "tts", "playai", "embed"))]
            prefer = [i for i in chat if "gpt-oss-120b" in i] or [i for i in chat if "llama" in i and "70b" in i] or chat
            if not prefer or attempt == 3:
                raise RuntimeError("Groq has no usable chat model for your key. Models offered: " + ", ".join(ids))
            GROQ_MODEL = prefer[0]
            print(f"[note] the old model name is gone; switching to {GROQ_MODEL}")

# A small file the agent will need to read later.
with open("notes.txt", "w") as f:
    f.write("Attend AI Expo: Nov 16\nSprint demo: Dec 2\nFinal report: Dec 9\n")
print("Created notes.txt")

# Checks your live answer (used in Part 4). Accepts "Nov 20", "November 20", "Nov. 20", etc.
def check_notes_answer(result):
    text = str(result)
    wanted = [("Nov", "November", "20"), ("Dec", "December", "2"), ("Dec", "December", "9")]
    found = [f"{a} {d}" for a, full, d in wanted
             if re.search(rf"\b(?:{a}|{full})\.?\s*{d}\b", text, re.I)]
    if len(found) == 3:
        print("PASS: the answer has all three real dates from notes.txt.")
    else:
        print(f"Not yet: found {len(found)} of the 3 real dates. Did the agent actually read notes.txt? Look at its steps.")


## Part 1. A prompt
A prompt is text you give a model so it writes text back. You ask once and get one answer. The model can only use what is in the prompt, so ask it to do something it has no way to do and watch what happens.

In [ ]:
# PART 1: a plain prompt. One question in, one answer out.
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Read the file notes.txt and list the three deadlines in it, with their dates."},
]
print(ask_groq(messages))


**Double-click this cell and answer in your own words (2 or 3 sentences):**
Did the model read `notes.txt`? How can you tell? What would it have needed in order to really answer?

*(your answer here)*

## Part 2. An agent
An agent is **a prompt plus three things**:
1. **Tools**: functions the model can ask for.
2. **A loop**: ask the model what to do next, run the tool for it, show it the result, ask again.
3. **A stopping rule**: the model says it is finished, or you stop it at a step limit.

The model never runs anything. **Your code runs the tools.** The model only asks, in a fixed format.

## Part 3. The challenge: get an AI to build your agent, then prove it works

Your job is to **write a good prompt** (use the 8-part structure from class) that gets an AI of your choice (Gemini in Colab, Claude, ChatGPT, anything) to write the Python function below. Then paste its code in, run the checks, and fix what fails, **by improving your prompt and asking again, or by fixing the code yourself**.

### The specification (the checks test exactly this)

```python
def run_agent(goal, tools, chat_fn, max_steps=6, verbose=False):
    ...
    return final_answer   # a string
```

* `goal`: a string, the job to do.
* `tools`: a dict. Each key is a tool name, each value is a function that takes **one string** and returns **one string**. Every tool function has a **docstring** that says what it does.
* `chat_fn(messages)`: sends a list of `{"role": ..., "content": ...}` dicts to a model and returns the reply as a string. Call it once per step.
* The model replies in one of two formats:
  * **To use a tool:** a line `ACTION: <tool name>` and then a line `INPUT: <text for the tool>`
  * **To finish:** a line starting `FINISH: <the final answer>`
* `run_agent` repeats: ask the model, run the tool it asked for, send the result back to the model in a message that starts `OBSERVATION: `, and ask again.
* It returns the final answer when the model finishes, or a string starting `STOPPED` when it reaches `max_steps`.
* **Whatever goes wrong must not crash or end the run.** Tell the model what went wrong in a message that contains the word `ERROR`: an unknown tool name, a tool that crashes, or a reply in neither format.
* The model must be told its goal, the **name and description (the docstring) of each tool**, and the reply format.
* **The model must not be allowed to finish before it has used a tool.** Small models often give up and say "I can't access files" on their first reply. If the model replies `FINISH` and no tool has been used yet (and there are tools), do not return. Send a message containing `ERROR` that tells it to use a tool first, and keep going.
* When `verbose=True`, print every step as it happens: the model's reply and the observation. This is how you watch your agent work.

### Scoring
Level 1 (3 checks): the loop. Level 2 (3 checks): robustness. Level 3 (3 checks + the live run + reflection): a real agent.

**Write your prompt here.** This cell is just where you keep it. Paste it into your AI.

In [ ]:
MY_PROMPT = """
(write your 8-part prompt here)
"""


**Paste the code your AI gave you here, then run the cell.**

In [ ]:
def run_agent(goal, tools, chat_fn, max_steps=6, verbose=False):
    raise NotImplementedError("Paste your agent here")


**Now run the checks.** Run this every time you change your agent.

In [ ]:
# THE CHECKS: run this cell every time you change your agent. Do not edit it.
# It runs YOUR run_agent against a pretend model (instant, free) and tells you what passes.
import signal, copy

class _Script:
    """A pretend model: hands back prepared replies one at a time and remembers what it was shown."""
    def __init__(self, *replies, repeat=None):
        self.replies, self.repeat, self.calls = list(replies), repeat, []
    def __call__(self, messages, *a, **k):
        self.calls.append(copy.deepcopy(messages))
        if len(self.calls) > 40:
            raise RuntimeError("the agent made more than 40 model calls")
        if self.replies:
            return self.replies.pop(0)
        return self.repeat if self.repeat is not None else "FINISH: (the script ran out)"

def _text(messages):
    return " ".join(str(m.get("content", "")) if isinstance(m, dict) else str(m) for m in messages)

class _TooSlow(Exception):
    pass

def _alarm(signum, frame):
    raise _TooSlow()

def _run(goal, tools, script, max_steps=6):
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(5)
    except Exception:
        pass
    try:
        return run_agent(goal, tools, script, max_steps=max_steps)
    finally:
        try:
            signal.alarm(0)
        except Exception:
            pass

def _echo(s): return "echoed: " + s
def _crash(s): raise ValueError("boom")

def c1():
    seen = []
    tools = {"echo": lambda s: seen.append(s) or "echoed: " + s}
    sc = _Script("ACTION: echo\nINPUT: hello", "FINISH: all done")
    out = _run("say hello", tools, sc)
    assert seen == ["hello"], "your agent did not run the tool with the INPUT text"
    assert str(out).strip() == "all done", f"run_agent should return the FINISH text; it returned {out!r}"
    assert len(sc.calls) == 2, f"expected 2 model calls, saw {len(sc.calls)}"

def c2():
    sc = _Script("ACTION: lookup\nINPUT: x", "FINISH: ok")
    _run("find it", {"lookup": lambda s: "SECRET-4711"}, sc)
    t = _text(sc.calls[1]) if len(sc.calls) > 1 else ""
    assert "SECRET-4711" in t, "the tool's result was never shown to the model"
    assert "OBSERVATION" in t, "the tool result should be sent back in a message starting OBSERVATION:"

def c3():
    sc = _Script(repeat="ACTION: echo\nINPUT: again")
    out = _run("loop", {"echo": _echo}, sc, max_steps=3)
    assert len(sc.calls) == 3, f"max_steps=3 should mean exactly 3 model calls; saw {len(sc.calls)}"
    assert str(out).upper().startswith("STOPPED"), f"at the limit, return text starting STOPPED; got {out!r}"

def c4():
    sc = _Script("ACTION: teleport\nINPUT: x", "ACTION: echo\nINPUT: y", "FINISH: ok")
    out = _run("go", {"echo": _echo}, sc)
    assert str(out).strip() == "ok", "an unknown tool should not end or crash the run"
    assert "ERROR" in _text(sc.calls[1]), "tell the model about the problem with a message containing ERROR"

def c5():
    sc = _Script("ACTION: bad\nINPUT: x", "FINISH: ok")
    out = _run("go", {"bad": _crash}, sc)
    assert str(out).strip() == "ok", "a crashing tool should not end or crash the run"
    assert "ERROR" in _text(sc.calls[1]), "tell the model the tool failed with a message containing ERROR"

def c6():
    sc = _Script("I think the answer is 42.", "ACTION: echo\nINPUT: y", "FINISH: ok")
    out = _run("go", {"echo": _echo}, sc)
    assert str(out).strip() == "ok", "a reply in the wrong format should not be taken as the final answer"
    assert len(sc.calls) >= 2 and "ERROR" in _text(sc.calls[1]), "tell the model its reply was in the wrong format (ERROR)"

def c7():
    sc = _Script("ACTION: echo\nINPUT: MARKER-77", "ACTION: echo\nINPUT: second", "FINISH: ok")
    _run("go", {"echo": _echo}, sc)
    assert len(sc.calls) == 3, f"expected 3 model calls; saw {len(sc.calls)}"
    assert "MARKER-77" in _text(sc.calls[2]) and "echoed: second" not in _text(sc.calls[0]), "the model should be shown the whole history on every call"
    assert "echoed: MARKER-77" in _text(sc.calls[2]), "the third call should still show the FIRST tool result"
    assert "INPUT: MARKER-77" in _text(sc.calls[2]), "the model should also be shown its own earlier replies, not just the tool results"

def _alpha(s):
    "ALPHA-DESCRIPTION reverses text"
    return s
def _beta(s):
    "BETA-DESCRIPTION counts words"
    return s

def c8():
    sc = _Script("FINISH: ok")
    _run("GOAL-TEXT-9", {"alpha_tool": _alpha, "beta_tool": _beta}, sc)
    t = _text(sc.calls[0])
    assert "GOAL-TEXT-9" in t, "the model must be given the goal"
    assert "alpha_tool" in t and "beta_tool" in t, "the model must be told the names of its tools"
    assert "ALPHA-DESCRIPTION" in t and "BETA-DESCRIPTION" in t, "the model must be told what each tool does (use the tool's docstring)"
    assert "ACTION" in t and "FINISH" in t, "the model must be told the reply format (ACTION / INPUT / FINISH)"

def c9():
    sc = _Script("FINISH: early", "ACTION: echo\nINPUT: a", "FINISH: done")
    out = _run("go", {"echo": _echo}, sc)
    assert str(out).strip() == "done", f"a FINISH before any tool was used should not end the run; it returned {out!r}"
    assert "ERROR" in _text(sc.calls[1]), "tell the model it cannot finish before using a tool (a message containing ERROR)"
    assert "echoed: a" in _text(sc.calls[2]), "after the model uses the tool, the tool result must be shown to it"

_LEVELS = [
    ("LEVEL 1 (the loop)", [("calls a tool, then finishes", c1), ("shows the tool result to the model", c2), ("stops at the step limit", c3)]),
    ("LEVEL 2 (robust)", [("survives an unknown tool", c4), ("survives a tool that crashes", c5), ("survives a reply in the wrong format", c6)]),
    ("LEVEL 3 (a real agent)", [("remembers the whole conversation", c7), ("tells the model its goal, tools (with descriptions) and format", c8), ("will not accept FINISH before a tool has been used", c9)]),
]

def run_checks():
    total = passed = 0
    for level, items in _LEVELS:
        got = 0
        print(level)
        for name, fn in items:
            total += 1
            try:
                fn()
                print(f"   PASS  {name}")
                got += 1
            except _TooSlow:
                print(f"   FAIL  {name}\n         it ran for more than 5 seconds (is there a loop that never ends?)")
            except (NameError, NotImplementedError):
                print(f"   FAIL  {name}\n         there is no agent yet. Paste your agent into the cell above and run that cell.")
            except AssertionError as e:
                print(f"   FAIL  {name}\n         {e}")
            except Exception as e:
                print(f"   FAIL  {name}\n         your agent crashed: {type(e).__name__}: {e}")
        passed += got
        print(f"   -> {got} of {len(items)}\n")
    print(f"TOTAL: {passed} of {total} checks pass")

run_checks()


## Part 4. Run it for real
The checks used a pretend model. Now give your agent a real one (`ask_groq`) and real tools.

**Required:** answer the question that the plain prompt could not. Write a tool that reads `notes.txt` (with a docstring!), give it to your agent, and ask for the three deadlines. Run it with `verbose=True` and watch the steps. Then check your answer.

**Then, on your own:** use `MY_TASK_IDEA` from the top. Your task must relate to one of the prompts we developed from Sommerville. Give your agent at least **two tools** that make sense for it (ask your AI to write them), run it, and watch each step.

If Groq asks you to slow down, the helper waits and tries again. That is normal on the free tier.

In [ ]:
# Your tools: each takes ONE string and returns ONE string.
tools = {}

goal = "Read notes.txt and list the three deadlines with their dates."

# Write a docstring in each tool: the model reads it to learn what the tool does.
# result = run_agent(goal, tools, ask_groq, max_steps=6, verbose=True)
# print(result)
# check_notes_answer(result)


## Part 5. Reflection (double-click each answer to type)

**1. What could your agent do that the plain prompt in Part 1 could not? Name a specific step you saw.**

*(your answer)*

**2. What did your AI get wrong the first time, and how did you find out? (The checks, the live run, or reading the code.)**

*(your answer)*

**3. Which sentence in your prompt made the biggest difference to the code you got back? Why?**

*(your answer)*

**4. Remove the Verification and Constraints parts from your prompt, ask the AI again, and run the checks on the new code. What changed, or did nothing change? Report what you saw, not what you expected.**

*(your answer)*

**5. Name one thing that went wrong, or could go wrong, that a better prompt could NOT fix.**

*(your answer)*

## Hints
**Hint 1 (nothing passes).** Did you paste the code into the cell above and *run* it? Is the function called exactly `run_agent` with exactly those four parameters?

**Hint 2 (Level 1 fails).** Read the failure message. A loop needs: call `chat_fn`, find `ACTION`/`INPUT` or `FINISH` in the reply, run the tool, add the *result* as a new message, and go around again. Did your prompt say all of that, including the exact message formats?

**Hint 2b (the live run says "I can't read files").** Read the printed steps. Did your agent tell the model the *description* of each tool, and refuse to accept FINISH before any tool was used? Those are in the specification.

**Hint 3 (Level 2 fails).** Your prompt probably did not say what to do when things go wrong. Tell the AI each bad situation and exactly what the agent should send back.